In [23]:
#Connect to MySQL
import mysql.connector

def connect_to_db():
    try:
        # configuration for your local mysql server
        connection = mysql.connector.connect(
            host="127.0.0.1",
            user="root",        # your mysql username
            password="09080904", # your mysql password
            database="sports_ticketing_db"
        )
        
        if connection.is_connected():
            print("successfully connected to the database")
            return connection

    except mysql.connector.Error as err:
        print(f"error: {err}")
        return None

connection = connect_to_db()

successfully connected to the database


In [44]:
# RUN IF WANTO DROP TABLE(S) OR DATABASE
connection.close()

In [28]:
# ============================================================
# INSERT STADIUM DATA
# ============================================================
stadium_data = [
    ("Vitality Stadium", "Bournemouth, UK", 11307, "vitalitymap.jpg"),
    ("Kenilworth Road", "Luton, UK", 12056, "kenilworthroad.jpg"),
    ("Gtech Community Stadium", "London, UK", 17250, "gtechcommunitymap.jpg"),
    ("Stamford Bridge", "London, UK", 40341, "stamfordbridgemap.jpg"),
    ("Craven Cottage", "London, UK", 25700, "cravencottagemap.jpg")
]

cursor = connection.cursor()

try:
    # Adding Unique Constraint first to prevent logic errors during sync
    print("Applying Unique Constraint to SEAT table...")
    cursor.execute("ALTER TABLE seat ADD UNIQUE KEY uq_stadium_seat (stadium_id, seat_name);")
    connection.commit()
except mysql.connector.Error as err:
    # Ignore if index already exists (Error 1061)
    if err.errno == 1061:
        print("  ! Unique key already exists. Skipping...")
    else:
        print(f"  ! Error: {err}")

# Seeding Stadiums
insert_stadium_sql = """
    INSERT INTO STADIUM (stadium_name, address, capacity, map_image_url)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE 
        address = VALUES(address),
        capacity = VALUES(capacity);
"""

try:
    cursor.executemany(insert_stadium_sql, stadium_data)
    connection.commit()
    print(f"✓ Successfully seeded {len(stadium_data)} stadiums.")
    
    # Verification Select
    cursor.execute("SELECT stadium_id, stadium_name FROM STADIUM;")
    for row in cursor.fetchall():
        print(f"  ID: {row[0]} | Name: {row[1]}")

except mysql.connector.Error as err:
    connection.rollback()
    print(f"✗ Stadium Seeding failed: {err}")
finally:
    cursor.close()

Applying Unique Constraint to SEAT table...
✓ Successfully seeded 5 stadiums.
  ID: 1 | Name: Vitality Stadium
  ID: 2 | Name: Kenilworth Road
  ID: 3 | Name: Gtech Community Stadium
  ID: 4 | Name: Stamford Bridge
  ID: 5 | Name: Craven Cottage


In [24]:
#Import library
import logging
from dataclasses import dataclass
from typing import Optional

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)


# =============================================================================
# ROW HELPERS — support multi-character rows: A–Z, AA–AZ, BA–BZ, ...
# =============================================================================

def row_to_index(row: str) -> int:
    """
    Converts a row label to a zero-based integer index.

    Input:
        row (str) : Row label, e.g. "A", "Z", "AA", "AB", "BA"

    Output:
        int : Zero-based index

    Examples:
        "A"  → 0
        "Z"  → 25
        "AA" → 26
        "AB" → 27
        "AZ" → 51
        "BA" → 52
    """
    row   = row.upper()
    index = 0
    for char in row:
        index = index * 26 + (ord(char) - ord('A') + 1)
    return index - 1


def index_to_row(index: int) -> str:
    """
    Converts a zero-based integer index to a row label.

    Input:
        index (int) : Zero-based index, e.g. 0, 25, 26

    Output:
        str : Row label, e.g. "A", "Z", "AA"

    Examples:
        0  → "A"
        25 → "Z"
        26 → "AA"
        51 → "AZ"
        52 → "BA"
    """
    label = ""
    n     = index + 1
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        label        = chr(ord('A') + remainder) + label
    return label


def rows_in_range(row_start: str, row_end: str) -> list[str]:
    """
    Returns all row labels between row_start and row_end inclusive.

    Input:
        row_start (str) : e.g. "A"
        row_end   (str) : e.g. "AB"

    Output:
        list[str] : e.g. ["A","B",...,"Z","AA","AB"]
    """
    start = row_to_index(row_start.upper())
    end   = row_to_index(row_end.upper())
    return [index_to_row(i) for i in range(start, end + 1)]

# =============================================================================
# ENCODING LOGIC
# =============================================================================

def get_zone_code(full_seat_zone: str) -> str:
    """
    Encodes the zone name into a short code for seat_name.
    Logic:
    1. Extract initials from words before the dash '-' or '–'.
    2. Map the type after the dash: VIP -> 1, Normal -> 2, Economy -> 3.
    Example: 'Main Stand – VIP' -> 'MS1'
    """
    # Replace long dash with standard dash and split
    parts = full_seat_zone.replace("–", "-").split("-")
    
    # Part 1: Stand Name (e.g., 'Main Stand')
    stand_part = parts[0].strip().upper()
    # Get first letter of each word: 'MAIN STAND' -> 'MS'
    prefix = "".join([w[0] for w in stand_part.split() if w])
    
    # Part 2: Seat Type Mapping
    suffix = "0" # Default if no match found
    if len(parts) > 1:
        type_part = parts[1].strip().upper()
        if "VIP" in type_part:
            suffix = "1"
        elif "NORMAL" in type_part:
            suffix = "2"
        elif "ECONOMY" in type_part:
            suffix = "3"
            
    return f"{prefix}{suffix}"

# =============================================================================
# ZONE CONFIGURATION
# =============================================================================

@dataclass
class ZoneConfiguration:
    """
    Defines one contiguous block of seats sharing the same seat type.

    Input:
        seat_zone     (str)        : Full name: e.g., "Main Stand – VIP"
        row_start     (str)        : First row, e.g. "A" or "AA"
        row_end       (str)        : Last row,  e.g. "Z" or "AB"
        seats_per_row (int)        : Number of seats per row, no upper limit
        seat_type     (str | None) : "vip" | "normal" | "economy" | None
    """

    seat_zone:     str   
    row_start:     str
    row_end:       str
    seats_per_row: int
    seat_type:     Optional[str] = None

    VALID_SEAT_TYPES = {"vip", "normal", "economy"}

    def __post_init__(self):
        self.row_start = self.row_start.upper()
        self.row_end   = self.row_end.upper()
        if self.seat_type:
            self.seat_type = self.seat_type.lower()

    def validate(self) -> tuple[bool, str]:
        if not self.row_start.isalpha():
            return False, f"row_start must be letters only, got '{self.row_start}'"
        if not self.row_end.isalpha():
            return False, f"row_end must be letters only, got '{self.row_end}'"
        if row_to_index(self.row_start) > row_to_index(self.row_end):
            return False, f"row_start '{self.row_start}' must be <= row_end '{self.row_end}'"
        if self.seats_per_row < 1:
            return False, "seats_per_row must be at least 1"
        if self.seat_type and self.seat_type not in self.VALID_SEAT_TYPES:
            return False, f"seat_type '{self.seat_type}' invalid; choose from {self.VALID_SEAT_TYPES}"
        return True, "ok"

    @property
    def capacity(self) -> int:
        return len(rows_in_range(self.row_start, self.row_end)) * self.seats_per_row

    def generate_seat_tuples(self, stadium_id: int) -> list[tuple]:
        """
        Output:
            List of tuples: (stadium_id, seat_row, seat_number, seat_zone, seat_name, seat_type)

        Example (rows A–B, seats_per_row=3):
            (1, "A", "01", "A01", "West1", "vip")
            (1, "A", "02", "A02", "West1","vip")
            (1, "A", "03", "A03", "West1","vip")
            (1, "B", "01", "B01", "West1","vip")
            ...
        """
        ok, msg = self.validate()
        if not ok:
            logger.error("[%s] Validation failed: %s", self.seat_zone, msg)
            return []

        pad    = len(str(self.seats_per_row))
        tuples = []

        # Get encoded short code (e.g., 'MS1')
        short_code = get_zone_code(self.seat_zone)

        for row_label in rows_in_range(self.row_start, self.row_end):
            for s_num in range(1, self.seats_per_row + 1):
                seat_number = str(s_num).zfill(pad)
                # Encoded seat name for tickets
                seat_name = f"{short_code}_{row_label}{seat_number}"
                tuples.append((
                    stadium_id,    # 0
                    row_label,     # 1
                    seat_number,   # 2
                    seat_name,     # 3
                    self.seat_zone, # 4
                    self.seat_type # 5
                ))

        return tuples

def sync_seats_to_db(
    connection,
    stadium_id: int,
    configs: list[ZoneConfiguration],
) -> None:
    """
    Upserts all seat records for a stadium into the SEAT table.
    Safe to re-run: existing seats update seat_type if changed,
    new seats are inserted.

    Requires a unique constraint on the SEAT table:
        ALTER TABLE seat ADD UNIQUE KEY uq_stadium_seat (stadium_id, seat_name);

    Input:
        connection : Active mysql.connector connection (already connected)
        stadium_id : Target stadium, e.g. 1
        configs    : List of ZoneConfiguration objects defining all seat blocks

    Output:
        None. Logs total seats synced on success, rolls back and raises on error.
    """
    all_tuples: list[tuple] = []

    for cfg in configs:
        rows = cfg.generate_seat_tuples(stadium_id)
        if rows:
            logger.info("  %-30s  rows %s–%s  %d seats  type=%s",
                        cfg.seat_zone, cfg.row_start, cfg.row_end,
                        len(rows), cfg.seat_type or "none")
        all_tuples.extend(rows)

    if not all_tuples:
        logger.warning("No valid seat data to insert — aborting.")
        return

    upsert_sql = """
        INSERT INTO seat (stadium_id, seat_row, seat_number, seat_name, seat_zone, seat_type)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            seat_zone = VALUES(seat_zone),
            seat_type = VALUES(seat_type);
    """

    cursor = connection.cursor()
    try:
        cursor.executemany(upsert_sql, all_tuples)
        connection.commit()
        logger.info("Synced %d seats for stadium_id=%d", len(all_tuples), stadium_id)
    except mysql.connector.Error as err:
        connection.rollback()
        logger.error("Seat sync failed: %s", err)
        raise
    finally:
        cursor.close()

def debug_configs(stadium_id: int, configs: list[ZoneConfiguration]) -> None:
    """
    Prints a full checkpoint report before touching the DB.
    Run this first to verify everything looks correct.
    """
    print("\n" + "=" * 60)
    print("  CHECKPOINT — Pre-sync validation")
    print("=" * 60)
    print(f"  Stadium ID   : {stadium_id}")
    print(f"  Total zones  : {len(configs)}")

    all_valid  = True
    grand_total = 0

    for i, cfg in enumerate(configs, 1):
        ok, msg = cfg.validate()
        status  = "✓" if ok else "✗"

        print(f"\n  [{status}] Zone {i}: {cfg.seat_zone}")
        print(f"       rows      : {cfg.row_start} → {cfg.row_end}  "
              f"({len(rows_in_range(cfg.row_start, cfg.row_end))} rows)")
        print(f"       seats/row : {cfg.seats_per_row}")
        print(f"       seat_type : {cfg.seat_type or 'NULL'}")
        print(f"       capacity  : {cfg.capacity} seats")

        if not ok:
            print(f"       ERROR     : {msg}")
            all_valid = False
            continue

        # Spot-check first and last seat name generated
        tuples = cfg.generate_seat_tuples(stadium_id)
        if tuples:
            first = tuples[0]
            last  = tuples[-1]
            # 0:stadium_id, 1:row, 2:num, 3:name, 4:zone, 5:type
            print(f"       first seat    : {first[3]} | zone: {first[4]} | type: {first[5] or 'NULL'}")
            print(f"       last seat     : {last[3]} | zone: {last[4]} | type: {last[5] or 'NULL'}")

        grand_total += cfg.capacity

    print("\n" + "-" * 60)
    print(f"  Total seats to insert : {grand_total}")
    print(f"  All configs valid     : {all_valid}")
    print("=" * 60)

    if not all_valid:
        print("\n  ⚠ Fix errors above before running sync_seats_to_db.\n")
    else:
        print("\n  ✓ Safe to run sync_seats_to_db.\n")

In [ ]:
#CONFIG FOR 5 STADIUMS (NUMBER OF SEAT SIMULATED BASE ON REAL SEATMAP)
# =============================================================================
# VITALITY STADIUM CONFIG — 11,370 seats, 4 zones
# =============================================================================
#
# Zone          Rows   Seats/Row   Total
# ------------ ------  ---------  ------
# Main  VIP    A–E    (5)  × 140 =   700
# Main  Normal F–Z   (21)  × 140 = 2,940  → 3,640
# East  VIP    A–E    (5)  × 120 =   600
# East  Normal F–Z   (21)  × 120 = 2,520  → 3,120
# North VIP    A–E    (5)  ×  88 =   440
# North Normal F–Z   (21)  ×  88 = 1,848  → 2,288
# South VIP    A–E    (5)  ×  88 =   440
# South Normal F–Z   (21)  ×  88 = 1,848  → 2,288
#                              TOTAL      11,336  (target 11,370, diff 34 = 0.3%)
#
# Now that rows are unlimited, can extend to AA onward if needed.
# =============================================================================

vitality_configs = [

    # Main Stand — South, widest (sections 1–9 on map)
    ZoneConfiguration("Main Stand – VIP",    "A", "E",  140, "vip"),
    ZoneConfiguration("Main Stand – Normal", "F", "Z",  140, "normal"),

    # East Stand — North (sections 16–24 on map)
    ZoneConfiguration("East Stand – VIP",    "A", "E",  120, "vip"),
    ZoneConfiguration("East Stand – Normal", "F", "Z",  120, "normal"),

    # North Stand — West, narrower (sections 10–15 on map)
    ZoneConfiguration("North Stand – VIP",    "A", "E", 88, "vip"),
    ZoneConfiguration("North Stand – Normal", "F", "Z", 88, "normal"),

    # South Stand — East, narrower (sections 25–30 on map)
    ZoneConfiguration("South Stand – VIP",    "A", "E", 88, "vip"),
    ZoneConfiguration("South Stand – Normal", "F", "Z", 88, "normal"),
]

# =============================================================================
# KENILWORTH ROAD — stadium_id = 2, capacity 12,056
# =============================================================================
# Zone                        Rows  Seats  Count   Total
# --------------------------  ----  -----  -----  ------
# Main Stand Upper VIP          5     95     1      475
# Main Stand Upper Normal      21     95     1    1,995  → Upper 2,470
# Main Stand Lower VIP          5     95     1      475
# Main Stand Lower Normal      21     95     1    1,995  → Lower 2,470  → MS 4,940
# Kenilworth Upper VIP          5     79     1      395
# Kenilworth Upper Normal      21     79     1    1,659  → Upper 2,054
# Kenilworth Lower VIP          5     79     1      395
# Kenilworth Lower Normal      21     79     1    1,659  → Lower 2,054  → KL 4,108
# Oak Road Left Normal         26     37     1      962
# Oak Road Right Normal        26     37     1      962                  → OR 1,924
# David Preece Left Normal     16     22     1      352
# David Preece Right Normal    16     22     1      352                  → DP   704
# Executive Left VIP            5     21     1      105
# Executive Right VIP           5     21     1      105                  → EB   210
#                                                        -----
#                                              TOTAL    11,886  (target 12,056, diff 170 = 1.4%)
# =============================================================================

kenilworth_configs = [

    # -------------------------------------------------------------------------
    # MAIN STAND (MS) — runs along the pitch, 2 tiers
    # Named MS on map. 26 rows × 95 seats × 2 tiers = 4,940
    # VIP  : A–E  (5 rows  = 19%)
    # Normal: F–Z (21 rows = 81%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("Main Stand Upper – VIP",    "A", "E", 95, "vip"),
    ZoneConfiguration("Main Stand Upper – Normal", "F", "Z", 95, "normal"),
    ZoneConfiguration("Main Stand Lower – VIP",    "A", "E", 95, "vip"),
    ZoneConfiguration("Main Stand Lower – Normal", "F", "Z", 95, "normal"),

    # -------------------------------------------------------------------------
    # KENILWORTH STAND (KL / KU on map) — behind goal, oldest stand
    # KL = lower, KU = upper. 26 rows × 79 seats × 2 tiers = 4,108
    # VIP  : A–E
    # Normal: F–Z
    # -------------------------------------------------------------------------
    ZoneConfiguration("Kenilworth Stand Upper – VIP",    "A", "E", 79, "vip"),
    ZoneConfiguration("Kenilworth Stand Upper – Normal", "F", "Z", 79, "normal"),
    ZoneConfiguration("Kenilworth Stand Lower – VIP",    "A", "E", 79, "vip"),
    ZoneConfiguration("Kenilworth Stand Lower – Normal", "F", "Z", 79, "normal"),

    # -------------------------------------------------------------------------
    # OAK ROAD END (OR) — away end, opposite Kenilworth
    # 26 rows × 37 seats × 2 sides = 1,924
    # All Normal (away fans, no VIP distinction)
    # -------------------------------------------------------------------------
    ZoneConfiguration("Oak Road End Left – Normal",  "A", "Z", 37, "normal"),
    ZoneConfiguration("Oak Road End Right – Normal", "A", "Z", 37, "normal"),

    # -------------------------------------------------------------------------
    # DAVID PREECE STAND (DP) — corner family stand
    # Named DP A–E on map. 16 rows × 22 seats × 2 sides = 704
    # All Normal (family stand)
    # -------------------------------------------------------------------------
    ZoneConfiguration("David Preece Left – Normal",  "A", "P", 22, "normal"),
    ZoneConfiguration("David Preece Right – Normal", "A", "P", 22, "normal"),

    # -------------------------------------------------------------------------
    # EXECUTIVE BOXES (EB) — premium boxes along one side
    # 5 rows × 21 seats × 2 sides = 210
    # All VIP
    # -------------------------------------------------------------------------
    ZoneConfiguration("Executive Boxes Left – VIP",  "A", "E", 21, "vip"),
    ZoneConfiguration("Executive Boxes Right – VIP", "A", "E", 21, "vip"),
]

# =============================================================================
# GTECH COMMUNITY STADIUM — stadium_id = 3, capacity 17,250
# =============================================================================
#
# Zone                  Rows      Seats/Row   Total
# --------------------  --------  ---------  ------
# North Stand VIP       A–K  (11)  × 200 =   2,200
# North Stand Normal    L–Z  (15)  × 200 =   3,000  → NS  5,200
# South Stand VIP       A–K  (11)  × 200 =   2,200
# South Stand Normal    L–Z  (15)  × 200 =   3,000  → SS  5,200
# West Stand  VIP       A–E   (5)  × 133 =     665
# West Stand  Normal    F–P  (11)  × 133 =   1,463
# West Stand  Economy   Q–Z  (10)  × 133 =   1,330  → WS  3,458
# East Stand  VIP       A–E   (5)  × 133 =     665
# East Stand  Normal    F–P  (11)  × 133 =   1,463
# East Stand  Economy   Q–Z  (10)  × 133 =   1,330  → ES  3,458
#                                                    ------
#                                          TOTAL    17,316  (target 17,250, diff 66 = 0.4%)
#
# Longside VIP  (N+S) : A–K = 11/26 rows = 42%  (~40% target)
# Shortside VIP (W+E) : A–E =  5/26 rows = 19%  (~20% target)
# Economy only on shortside (West + East), rows Q–Z
# =============================================================================

gtech_configs = [

    # -------------------------------------------------------------------------
    # NORTH STAND (NS) — longside, runs along the pitch, north side
    # 26 rows × 200 seats = 5,200
    # VIP    : A–K (11 rows, 42%)
    # Normal : L–Z (15 rows, 58%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("North Stand – VIP",    "A", "K", 200, "vip"),
    ZoneConfiguration("North Stand – Normal", "L", "Z", 200, "normal"),

    # -------------------------------------------------------------------------
    # SOUTH STAND (SS) — longside, opposite North, south side
    # 26 rows × 200 seats = 5,200  (same geometry as North)
    # VIP    : A–K (11 rows, 42%)
    # Normal : L–Z (15 rows, 58%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("South Stand – VIP",    "A", "K", 200, "vip"),
    ZoneConfiguration("South Stand – Normal", "L", "Z", 200, "normal"),

    # -------------------------------------------------------------------------
    # WEST STAND (WS) — shortside, behind west goal
    # 26 rows × 133 seats = 3,458
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("West Stand – VIP",     "A", "E", 133, "vip"),
    ZoneConfiguration("West Stand – Normal",  "F", "P", 133, "normal"),
    ZoneConfiguration("West Stand – Economy", "Q", "Z", 133, "economy"),

    # -------------------------------------------------------------------------
    # EAST STAND (ES) — shortside, behind east goal
    # 26 rows × 133 seats = 3,458  (same geometry as West)
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("East Stand – VIP",     "A", "E", 133, "vip"),
    ZoneConfiguration("East Stand – Normal",  "F", "P", 133, "normal"),
    ZoneConfiguration("East Stand – Economy", "Q", "Z", 133, "economy"),
]

# =============================================================================
# STAMFORD BRIDGE — stadium_id = 4, capacity 40,341
# =============================================================================
#
# Zone                        Rows      Seats/Row   Total
# --------------------------  --------  ---------  ------
# West Stand  VIP             A–M  (13)  × 500 =   6,500
# West Stand  Normal          N–Z  (13)  × 500 =   6,500  → WS 13,000
# East Stand  VIP             A–M  (13)  × 480 =   6,240
# East Stand  Normal          N–Z  (13)  × 480 =   6,240  → ES 12,480
# Matthew Harding VIP         A–E   (5)  × 280 =   1,400
# Matthew Harding Normal      F–P  (11)  × 280 =   3,080
# Matthew Harding Economy     Q–Z  (10)  × 280 =   2,800  → MH  7,280
# Shed End    VIP             A–E   (5)  × 292 =   1,460
# Shed End    Normal          F–P  (11)  × 292 =   3,212
# Shed End    Economy         Q–Z  (10)  × 292 =   2,920  → SH  7,592
#                                                          ------
#                                              TOTAL      40,352  (target 40,341, diff 11 = 0.03%)
#
# Longside VIP  (W+E) : A–M = 13/26 rows = 50%  (yêu cầu 50/50)
# Shortside VIP (MH+SH): A–E =  5/26 rows = 19%  (~20% target)
# Economy only on shortside (Matthew Harding + Shed End), rows Q–Z
# Shed End larger than MH 1 (292 vs 280)
# =============================================================================

stamford_configs = [

    # -------------------------------------------------------------------------
    # WEST STAND (WS) — longside, dọc sân phía tây
    # Gồm WU (upper) + WL (lower) + Millennium Suites + named boxes
    # 26 rows × 500 seats = 13,000
    # VIP    : A–M (13 rows, 50%) — upper tier + premium boxes
    # Normal : N–Z (13 rows, 50%) — lower tier
    # -------------------------------------------------------------------------
    ZoneConfiguration("West Stand – VIP",    "A", "M", 500, "vip"),
    ZoneConfiguration("West Stand – Normal", "N", "Z", 500, "normal"),

    # -------------------------------------------------------------------------
    # EAST STAND (ES) — longside, dọc sân phía đông
    # Gồm EU1–EU4 + executive sections (ELS, ELN) + named boxes
    # 26 rows × 480 seats = 12,480
    # VIP    : A–M (13 rows, 50%)
    # Normal : N–Z (13 rows, 50%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("East Stand – VIP",    "A", "M", 480, "vip"),
    ZoneConfiguration("East Stand – Normal", "N", "Z", 480, "normal"),

    # -------------------------------------------------------------------------
    # MATTHEW HARDING STAND (MH) — shortside phía đông, home end
    # Gồm L08–L16 (lower) + U08–U18 (upper)
    # 26 rows × 280 seats = 7,280
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("Matthew Harding Stand – VIP",     "A", "E", 280, "vip"),
    ZoneConfiguration("Matthew Harding Stand – Normal",  "F", "P", 280, "normal"),
    ZoneConfiguration("Matthew Harding Stand – Economy", "Q", "Z", 280, "economy"),

    # -------------------------------------------------------------------------
    # SHED END (SH) — shortside phía tây, away end
    # Gồm SU1–SU7 (upper) + SL1–SL7 (lower) + away section SU1/SL1
    # 26 rows × 292 seats = 7,592
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("Shed End – VIP",     "A", "E", 292, "vip"),
    ZoneConfiguration("Shed End – Normal",  "F", "P", 292, "normal"),
    ZoneConfiguration("Shed End – Economy", "Q", "Z", 292, "economy"),
]

# =============================================================================
# CRAVEN COTTAGE — stadium_id = 5, capacity 25,700
# =============================================================================
#
# Zone                      Rows      Seats/Row   Total
# ------------------------  --------  ---------  ------
# Johnny Haynes VIP         A–M  (13)  × 308 =   4,004
# Johnny Haynes Normal      N–Z  (13)  × 308 =   4,004  → JH  8,008
# Riverside Stand VIP       A–M  (13)  × 370 =   4,810
# Riverside Stand Normal    N–Z  (13)  × 370 =   4,810  → RS  9,620
# Hammersmith End VIP       A–E   (5)  × 158 =     790
# Hammersmith End Normal    F–P  (11)  × 158 =   1,738
# Hammersmith End Economy   Q–Z  (10)  × 158 =   1,580  → HE  4,108
# Putney End VIP            A–E   (5)  × 152 =     760
# Putney End Normal         F–P  (11)  × 152 =   1,672
# Putney End Economy        Q–Z  (10)  × 152 =   1,520  → PE  3,952
#                                                        ------
#                                            TOTAL      25,688  (target 25,700, diff 12 = 0.05%)
#
# Longside VIP  (JH+RS) : A–M = 13/26 rows = 50%  (50/50 split)
# Shortside VIP (HE+PE) : A–E =  5/26 rows = 19%  (~20% target)
# Economy only on shortside (Hammersmith + Putney), rows Q–Z
# Putney End smaller than Hammersmith (152 vs 158) — P5–P7 away section
# Riverside larger than Johnny Haynes (370 vs 308) — stand build 2021
# =============================================================================

craven_configs = [

    # -------------------------------------------------------------------------
    # JOHNNY HAYNES STAND (JH) — longside phía bắc, stand lịch sử
    # Gồm sections A–K + AL–KL (2 tiers)
    # 26 rows × 308 seats = 8,008
    # VIP    : A–M (13 rows, 50%)
    # Normal : N–Z (13 rows, 50%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("Johnny Haynes Stand – VIP",    "A", "M", 308, "vip"),
    ZoneConfiguration("Johnny Haynes Stand – Normal", "N", "Z", 308, "normal"),

    # -------------------------------------------------------------------------
    # RIVERSIDE STAND (RS) — longside phía nam, stand mới (2021)
    # Gồm R01–R08 + R11/R12/R17/R18 (corner extensions)
    # 26 rows × 370 seats = 9,620
    # VIP    : A–M (13 rows, 50%)
    # Normal : N–Z (13 rows, 50%)
    # -------------------------------------------------------------------------
    ZoneConfiguration("Riverside Stand – VIP",    "A", "M", 370, "vip"),
    ZoneConfiguration("Riverside Stand – Normal", "N", "Z", 370, "normal"),

    # -------------------------------------------------------------------------
    # HAMMERSMITH END (HE) — shortside phía tây, home end
    # Gồm H1–H8
    # 26 rows × 158 seats = 4,108
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("Hammersmith End – VIP",     "A", "E", 158, "vip"),
    ZoneConfiguration("Hammersmith End – Normal",  "F", "P", 158, "normal"),
    ZoneConfiguration("Hammersmith End – Economy", "Q", "Z", 158, "economy"),

    # -------------------------------------------------------------------------
    # PUTNEY END (PE) — shortside phía đông, away end
    # Gồm P1–P4 (home) + P5–P7 (away fans — orange trên map)
    # 26 rows × 152 seats = 3,952
    # VIP     : A–E  ( 5 rows, 19%)
    # Normal  : F–P  (11 rows, 42%)
    # Economy : Q–Z  (10 rows, 38%) — economy only on shortside
    # -------------------------------------------------------------------------
    ZoneConfiguration("Putney End – VIP",     "A", "E", 152, "vip"),
    ZoneConfiguration("Putney End – Normal",  "F", "P", 152, "normal"),
    ZoneConfiguration("Putney End – Economy", "Q", "Z", 152, "economy"),
]


In [29]:
#Check point
all_stadium_configs = {
    1: ("Vitality Stadium",           vitality_configs),
    2: ("Kenilworth Road",            kenilworth_configs),
    3: ("Gtech Community Stadium",    gtech_configs),
    4: ("Stamford Bridge",            stamford_configs),
    5: ("Craven Cottage",             craven_configs),
}

for stadium_id, (name, configs) in all_stadium_configs.items():
    print(f"\n{'='*60}")
    print(f"  {name}  (stadium_id={stadium_id})")
    debug_configs(stadium_id, configs)


  Vitality Stadium  (stadium_id=1)

  CHECKPOINT — Pre-sync validation
  Stadium ID   : 1
  Total zones  : 8

  [✓] Zone 1: Main Stand – VIP
       rows      : A → E  (5 rows)
       seats/row : 140
       seat_type : vip
       capacity  : 700 seats
       first seat    : MS1_A001 | zone: Main Stand – VIP | type: vip
       last seat     : MS1_E140 | zone: Main Stand – VIP | type: vip

  [✓] Zone 2: Main Stand – Normal
       rows      : F → Z  (21 rows)
       seats/row : 140
       seat_type : normal
       capacity  : 2940 seats
       first seat    : MS2_F001 | zone: Main Stand – Normal | type: normal
       last seat     : MS2_Z140 | zone: Main Stand – Normal | type: normal

  [✓] Zone 3: East Stand – VIP
       rows      : A → E  (5 rows)
       seats/row : 120
       seat_type : vip
       capacity  : 600 seats
       first seat    : ES1_A001 | zone: East Stand – VIP | type: vip
       last seat     : ES1_E120 | zone: East Stand – VIP | type: vip

  [✓] Zone 4: East Stand – No

In [30]:
#Save to Database
sync_seats_to_db(connection, stadium_id=1, configs=vitality_configs)
sync_seats_to_db(connection, stadium_id=2, configs=kenilworth_configs)
sync_seats_to_db(connection, stadium_id=3, configs=gtech_configs)
sync_seats_to_db(connection, stadium_id=4, configs=stamford_configs)
sync_seats_to_db(connection, stadium_id=5, configs=craven_configs)

INFO:   Main Stand – VIP                rows A–E  700 seats  type=vip
INFO:   Main Stand – Normal             rows F–Z  2940 seats  type=normal
INFO:   East Stand – VIP                rows A–E  600 seats  type=vip
INFO:   East Stand – Normal             rows F–Z  2520 seats  type=normal
INFO:   North Stand – VIP               rows A–E  440 seats  type=vip
INFO:   North Stand – Normal            rows F–Z  1848 seats  type=normal
INFO:   South Stand – VIP               rows A–E  440 seats  type=vip
INFO:   South Stand – Normal            rows F–Z  1848 seats  type=normal
INFO: Synced 11336 seats for stadium_id=1
INFO:   Main Stand Upper – VIP          rows A–E  475 seats  type=vip
INFO:   Main Stand Upper – Normal       rows F–Z  1995 seats  type=normal
INFO:   Main Stand Lower – VIP          rows A–E  475 seats  type=vip
INFO:   Main Stand Lower – Normal       rows F–Z  1995 seats  type=normal
INFO:   Kenilworth Stand Upper – VIP    rows A–E  395 seats  type=vip
INFO:   Kenilworth Stand

In [31]:
# ============================================================
# CHECK — Seats imported per stadium
# ============================================================
cursor = connection.cursor(dictionary=True)
cursor.execute("""
    SELECT
        s.stadium_id,
        s.stadium_name,
        COUNT(seat.seat_id)              AS total_seats,
        SUM(seat.seat_type = 'vip')      AS vip,
        SUM(seat.seat_type = 'normal')   AS normal,
        SUM(seat.seat_type = 'economy')  AS economy
    FROM STADIUM s
    LEFT JOIN SEAT seat ON seat.stadium_id = s.stadium_id
    GROUP BY s.stadium_id, s.stadium_name
    ORDER BY s.stadium_id
""")
for row in cursor.fetchall():
    # Set to 0 if None
    vip_count = row["vip"] or 0
    normal_count = row["normal"] or 0
    economy_count = row["economy"] or 0
    total = row["total_seats"] or 0

    print(f"  [{row['stadium_id']}] {row['stadium_name']:<30} "
          f"total={total:>6,}  "
          f"vip={vip_count:>5,}  "
          f"normal={normal_count:>6,}  "
          f"economy={economy_count:>6,}")
cursor.close()

  [1] Vitality Stadium               total=11,336  vip=2,180  normal= 9,156  economy=     0
  [2] Kenilworth Road                total=11,886  vip=1,950  normal= 9,936  economy=     0
  [3] Gtech Community Stadium        total=17,316  vip=5,730  normal= 8,926  economy= 2,660
  [4] Stamford Bridge                total=40,352  vip=15,600  normal=19,032  economy= 5,720
  [5] Craven Cottage                 total=25,688  vip=10,364  normal=12,224  economy= 3,100


True

In [32]:
# ============================================================
# INSERT EVENTS
# ============================================================
# 5 stadium × 4-5 events each = ~22 events
# ============================================================

events = [
    # stadium_id, event_name, event_date, sale_open_at, sale_close_at, status, pricing_config
    # --- Vitality Stadium (1) ---
    (1, "Bournemouth vs Arsenal",       "2025-08-16 15:00:00", "2025-07-01 09:00:00", "2025-08-16 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":80,"modifier_pct":20}]'),
    (1, "Bournemouth vs Liverpool",     "2025-09-20 17:30:00", "2025-08-01 09:00:00", "2025-09-20 14:30:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":75,"modifier_pct":25}]'),
    (1, "Bournemouth vs Wolves",        "2025-11-01 15:00:00", "2025-09-15 09:00:00", "2025-11-01 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-5}]'),
    (1, "Bournemouth vs Man City",      "2025-12-26 17:30:00", "2025-11-01 09:00:00", "2025-12-26 14:30:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"Hot Selling","type":"velocity_1h","threshold":40,"modifier_pct":15},{"name":"High Demand","type":"sold_pct","threshold":85,"modifier_pct":30}]'),
    (1, "Bournemouth vs Chelsea",       "2026-02-14 15:00:00", "2025-12-01 09:00:00", "2026-02-14 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10}]'),

    # --- Kenilworth Road (2) ---
    (2, "Luton vs Stoke City",          "2025-08-09 15:00:00", "2025-07-01 09:00:00", "2025-08-09 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-8}]'),
    (2, "Luton vs Sheffield Wed",       "2025-10-04 15:00:00", "2025-08-15 09:00:00", "2025-10-04 12:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":70,"modifier_pct":15}]'),
    (2, "Luton vs Burnley",             "2025-11-29 15:00:00", "2025-10-01 09:00:00", "2025-11-29 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":80,"modifier_pct":20}]'),
    (2, "Luton vs Derby County",        "2026-01-17 15:00:00", "2025-11-15 09:00:00", "2026-01-17 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-5}]'),

    # --- Gtech Community Stadium (3) ---
    (3, "Brentford vs Tottenham",       "2025-08-23 17:30:00", "2025-07-10 09:00:00", "2025-08-23 14:30:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":80,"modifier_pct":20}]'),
    (3, "Brentford vs Aston Villa",     "2025-10-18 15:00:00", "2025-09-01 09:00:00", "2025-10-18 12:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":75,"modifier_pct":15}]'),
    (3, "Brentford vs Newcastle",       "2025-12-06 15:00:00", "2025-10-15 09:00:00", "2025-12-06 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"Hot Selling","type":"velocity_1h","threshold":50,"modifier_pct":10}]'),
    (3, "Brentford vs West Ham",        "2026-01-31 15:00:00", "2025-12-01 09:00:00", "2026-01-31 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-8}]'),
    (3, "Brentford vs Everton",         "2026-03-07 15:00:00", "2026-01-10 09:00:00", "2026-03-07 12:00:00", "Draft",
     '[]'),

    # --- Stamford Bridge (4) ---
    (4, "Chelsea vs Man Utd",           "2025-08-30 17:30:00", "2025-07-15 09:00:00", "2025-08-30 14:30:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":85,"modifier_pct":30}]'),
    (4, "Chelsea vs Real Madrid (UCL)", "2025-10-22 21:00:00", "2025-09-01 09:00:00", "2025-10-22 18:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":60,"modifier_pct":40},{"name":"Hot Selling","type":"velocity_1h","threshold":100,"modifier_pct":20}]'),
    (4, "Chelsea vs Brighton",          "2025-12-13 15:00:00", "2025-10-20 09:00:00", "2025-12-13 12:00:00", "SoldOut",
     '[{"name":"High Demand","type":"sold_pct","threshold":70,"modifier_pct":25}]'),
    (4, "Chelsea vs Nottm Forest",      "2026-01-10 17:30:00", "2025-11-15 09:00:00", "2026-01-10 14:30:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":80,"modifier_pct":20}]'),
    (4, "Chelsea vs AC Milan (UCL)",    "2026-02-18 21:00:00", "2025-12-15 09:00:00", "2026-02-18 18:00:00", "OnSale",
     '[{"name":"High Demand","type":"sold_pct","threshold":50,"modifier_pct":35},{"name":"Hot Selling","type":"velocity_1h","threshold":80,"modifier_pct":15}]'),

    # --- Craven Cottage (5) ---
    (5, "Fulham vs Leicester",          "2025-08-16 15:00:00", "2025-07-01 09:00:00", "2025-08-16 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-8}]'),
    (5, "Fulham vs Ipswich",            "2025-10-25 15:00:00", "2025-09-10 09:00:00", "2025-10-25 12:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":75,"modifier_pct":15}]'),
    (5, "Fulham vs Crystal Palace",     "2025-12-20 15:00:00", "2025-11-01 09:00:00", "2025-12-20 12:00:00", "OnSale",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10},{"name":"High Demand","type":"sold_pct","threshold":80,"modifier_pct":20}]'),
    (5, "Fulham vs Wolves",             "2026-02-07 15:00:00", "2025-12-15 09:00:00", "2026-02-07 12:00:00", "Draft",
     '[]'),

    # --- Vitality Stadium (1) — Boxing ---
    (1, "WBO Cruiserweight: Smith vs Johnson",  "2025-09-27 19:00:00", "2025-08-01 09:00:00", "2025-09-27 17:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":70,"modifier_pct":25}]'),

    # --- Stamford Bridge (4) — Rugby + Charity ---
    (4, "Soccer Aid 2025",                      "2025-06-15 17:00:00", "2025-04-01 09:00:00", "2025-06-15 14:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":30,"modifier_pct":-10}]'),
    (4, "Rugby League: London vs Leeds",        "2025-07-20 15:00:00", "2025-06-01 09:00:00", "2025-07-20 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-8}]'),

    # --- Gtech Community Stadium (3) — Rugby + International ---
    (3, "London Irish vs Harlequins (Rugby)",   "2025-09-06 15:00:00", "2025-07-15 09:00:00", "2025-09-06 12:00:00", "Finished",
     '[{"name":"Early Bird","type":"days_before","threshold":21,"modifier_pct":-5}]'),
    (3, "Unity Cup 2025: Nigeria vs Jamaica",   "2025-10-11 17:00:00", "2025-08-20 09:00:00", "2025-10-11 14:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":75,"modifier_pct":15}]'),
    (3, "Unity Cup 2025: Ghana vs Trinidad",    "2025-10-14 19:00:00", "2025-08-20 09:00:00", "2025-10-14 16:00:00", "Finished",
     '[{"name":"High Demand","type":"sold_pct","threshold":75,"modifier_pct":15}]'),
]

insert_event_sql = """
    INSERT INTO EVENT (stadium_id, event_name, event_date, sale_open_at, sale_close_at, status, pricing_config)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor = connection.cursor()
cursor.executemany(insert_event_sql, events)
connection.commit()
print(f"✓ Inserted {cursor.rowcount} events")
cursor.close()

✓ Inserted 29 events


True

In [33]:
# ============================================================
# INSERT EVENT_ZONE
# ============================================================
# Base price is set per event.
# Same stadium can have different prices for derby vs regular match.
# Pricing rules (velocity, sold_pct) adjust on top of this base.
# ============================================================

# event_id → {seat_type: (zone_name, base_price)}
# Prices reflect match importance: derby/UCL > regular PL > cup/friendly
event_zone_config = {

    # --- Vitality Stadium ---
    # Bournemouth vs Arsenal — big away side, high demand
    1:  {"vip": ("Premium", 60.00),  "normal": ("Standard",  28.00)},
    # Bournemouth vs Liverpool — top 6 visitor
    2:  {"vip": ("Premium", 56.00),  "normal": ("Standard",  26.00)},
    # Bournemouth vs Wolves — mid-table, lower price
    3:  {"vip": ("Premium", 36.00),  "normal": ("Standard",  18.00)},
    # Bournemouth vs Man City — title contender
    4:  {"vip": ("Premium", 64.00),  "normal": ("Standard",  32.00)},
    # Bournemouth vs Chelsea — London club, good away support
    5:  {"vip": ("Premium", 52.00),  "normal": ("Standard",  24.00)},
    # Boxing — WBO Cruiserweight, one-off event, premium pricing
    6:  {"vip": ("Ringside", 120.00), "normal": ("General",  48.00)},

    # --- Kenilworth Road ---
    # Luton vs Stoke — Championship regular
    7:  {"vip": ("Executive", 24.00), "normal": ("Terrace",  12.00)},
    # Luton vs Sheffield Wed — mid-table
    8:  {"vip": ("Executive", 24.00), "normal": ("Terrace",  12.00)},
    # Luton vs Burnley — promotion rival, higher demand
    9:  {"vip": ("Executive", 36.00), "normal": ("Terrace",  18.00)},
    # Luton vs Derby — decent fixture
    10: {"vip": ("Executive", 28.00), "normal": ("Terrace",  14.00)},

    # --- Gtech Community Stadium ---
    # Brentford vs Tottenham — London derby
    11: {"vip": ("Club Level", 80.00), "normal": ("General", 36.00), "economy": ("Family Zone", 16.00)},
    # Brentford vs Aston Villa — Europa rival
    12: {"vip": ("Club Level", 64.00), "normal": ("General", 30.00), "economy": ("Family Zone", 14.00)},
    # Brentford vs Newcastle — top half clash
    13: {"vip": ("Club Level", 60.00), "normal": ("General", 28.00), "economy": ("Family Zone", 12.80)},
    # Brentford vs West Ham — London derby
    14: {"vip": ("Club Level", 72.00), "normal": ("General", 32.00), "economy": ("Family Zone", 15.20)},
    # Brentford vs Everton — Draft, placeholder price
    15: {"vip": ("Club Level", 48.00), "normal": ("General", 24.00), "economy": ("Family Zone", 11.20)},
    # London Irish vs Harlequins — Premiership Rugby derby
    16: {"vip": ("Club Level", 56.00), "normal": ("General", 26.00), "economy": ("Family Zone", 12.00)},
    # Unity Cup: Nigeria vs Jamaica
    17: {"vip": ("Club Level", 40.00), "normal": ("General", 20.00), "economy": ("Family Zone", 10.00)},
    # Unity Cup: Ghana vs Trinidad
    18: {"vip": ("Club Level", 40.00), "normal": ("General", 20.00), "economy": ("Family Zone", 10.00)},

    # --- Stamford Bridge ---
    # Chelsea vs Man Utd — biggest PL fixture
    19: {"vip": ("Hospitality", 200.00), "normal": ("General", 80.00),  "economy": ("Away Zone", 36.00)},
    # Chelsea vs Real Madrid UCL — European night, highest price
    20: {"vip": ("Hospitality", 320.00), "normal": ("General", 140.00), "economy": ("Away Zone", 60.00)},
    # Chelsea vs Brighton — regular PL
    21: {"vip": ("Hospitality", 120.00), "normal": ("General", 48.00),  "economy": ("Away Zone", 24.00)},
    # Chelsea vs Nottm Forest — regular PL
    22: {"vip": ("Hospitality", 112.00), "normal": ("General", 44.00),  "economy": ("Away Zone", 22.00)},
    # Chelsea vs AC Milan UCL — European night
    23: {"vip": ("Hospitality", 280.00), "normal": ("General", 120.00), "economy": ("Away Zone", 48.00)},
    # Soccer Aid — charity match, accessible pricing
    24: {"vip": ("Hospitality", 80.00),  "normal": ("General", 32.00),  "economy": ("Away Zone", 16.00)},
    # Rugby League: London vs Leeds
    25: {"vip": ("Hospitality", 60.00),  "normal": ("General", 28.00),  "economy": ("Away Zone", 14.00)},

    # --- Craven Cottage ---
    # Fulham vs Leicester — PL regular
    26: {"vip": ("River View", 72.00), "normal": ("General", 32.00), "economy": ("Riverside End", 15.20)},
    # Fulham vs Ipswich — lower mid-table
    27: {"vip": ("River View", 56.00), "normal": ("General", 26.00), "economy": ("Riverside End", 12.00)},
    # Fulham vs Crystal Palace — London derby
    28: {"vip": ("River View", 80.00), "normal": ("General", 36.00), "economy": ("Riverside End", 16.80)},
    # Fulham vs Wolves — Draft, placeholder
    29: {"vip": ("River View", 48.00), "normal": ("General", 22.00), "economy": ("Riverside End", 10.40)},
}

zone_rows = []
for event_id, zones in event_zone_config.items():
    for seat_type, (zone_name, base_price) in zones.items():
        zone_rows.append((event_id, seat_type, zone_name, base_price))

insert_zone_sql = """
    INSERT IGNORE INTO EVENT_ZONE (event_id, seat_type, zone_name, base_price)
    VALUES (%s, %s, %s, %s)
"""

cursor = connection.cursor()
cursor.executemany(insert_zone_sql, zone_rows)
connection.commit()
print(f"✓ Inserted {cursor.rowcount} event zones")
cursor.close()

✓ Inserted 77 event zones


True

In [34]:
# ============================================================
# POPULATE EVENT_SEAT 
# ============================================================
# Copy active seats of stadium -> EVENT_SEAT
# Link each seat to its corresponding EVENT_ZONE via seat_type
# Only for events with status != 'Draft'
# ============================================================

cursor = connection.cursor()

sync_query = """
    INSERT IGNORE INTO EVENT_SEAT (event_id, seat_id, zone_id)
    SELECT 
        e.event_id, 
        s.seat_id, 
        ez.zone_id
    FROM EVENT e
    JOIN SEAT s ON s.stadium_id = e.stadium_id
    JOIN EVENT_ZONE ez ON e.event_id = ez.event_id 
                       AND s.seat_type = ez.seat_type
    WHERE s.is_active = 1
    AND   e.status != 'Draft'
"""

try:
    cursor.execute(sync_query)
    connection.commit()
    print(f"✓ Populated {cursor.rowcount} event_seat rows with correct zone_id")
except Exception as e:
    connection.rollback()
    print(f"✗ Error populating event_seat: {e}")
finally:
    cursor.close()

✓ Populated 593640 event_seat rows with correct zone_id


In [35]:
# ============================================================
# CELL — VERIFY
# ============================================================
cursor = connection.cursor()
for table in ["EVENT", "EVENT_ZONE", "EVENT_SEAT"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:<15} {cursor.fetchone()[0]:>8,} rows")
cursor.close()

  EVENT                 29 rows
  EVENT_ZONE            77 rows
  EVENT_SEAT       593,640 rows


True

In [41]:
# ============================================================
# INSERT BOX OFFICE DATA 
# ============================================================

box_offices = [
    # Format: (name, type, stadium_id, commission_rate, address)
    
    # ONLINE CHANNELS (stadium_id = None)
    ("Official Website", "Online", None, 0.00, "https://tickets.sports.com"),

    # THIRD-PARTY AGENTS (stadium_id = None)
    ("Ticketmaster UK", "Agent", None, 10.00, "London HQ"),
    ("StubHub UK",      "Agent", None, 12.50, "Online Agent"),
    ("Viagogo",         "Agent", None, 15.00, "Online Marketplace"),
    ("See Tickets",     "Agent", None, 8.00,  "Nottingham, UK"),

    # VENUE BOX OFFICES (Linked to specific stadiums)
    ("Vitality Main Box Office",  "Venue", 1, 0.00, "Kings Park, Bournemouth"),
    ("Vitality East Gate Booth",  "Venue", 1, 0.00, "Kings Park, Bournemouth"),
    ("Kenilworth Road Ticket Office", "Venue", 2, 0.00, "1 Maple Rd, Luton"),
    ("Gtech South Stand Office",  "Venue", 3, 0.00, "Lionel Rd S, Brentford"),
    ("Stamford Bridge West Stand","Venue", 4, 0.00, "Fulham Rd, London"),
    ("Craven Cottage Riverside",  "Venue", 5, 0.00, "Stevenage Rd, London")
]

cursor = connection.cursor()

# Ensure the commission_rate column is properly sized
cursor.execute("ALTER TABLE BOX_OFFICE MODIFY COLUMN commission_rate DECIMAL(5,2) DEFAULT 0.00;")

insert_bo_sql = """
    INSERT INTO BOX_OFFICE (office_name, office_type, stadium_id, commission_rate, address, is_active)
    VALUES (%s, %s, %s, %s, %s, 1)
"""

try:
    # Logic: Unpacking exactly 5 values to match the tuple structure above
    data = [(name, t, sid, rate, addr) for name, t, sid, rate, addr in box_offices]
    cursor.executemany(insert_bo_sql, data)
    connection.commit()
    print(f"✓ Successfully inserted {len(box_offices)} box offices.")
except Exception as e:
    connection.rollback()
    print(f"✗ Error during Box Office seeding: {e}")
finally:
    cursor.close()

✓ Successfully inserted 11 box offices.


In [ ]:
# ============================================================
# CELL — GENERATE CUSTOMERS WITH FAKER
# ============================================================
from faker import Faker
import hashlib, secrets, random

fake = Faker('en_GB')
Faker.seed(42)

cursor = connection.cursor()

cursor.execute("SELECT email FROM CUSTOMER WHERE email IS NOT NULL")
existing_emails = {row[0] for row in cursor.fetchall()}
cursor.execute("SELECT phone_number FROM CUSTOMER WHERE phone_number IS NOT NULL")
existing_phones = {row[0] for row in cursor.fetchall()}

def fake_bcrypt(name):
    salt = secrets.token_hex(8)
    return "$2b$12$" + hashlib.sha256((name+salt).encode()).hexdigest()[:53]

def unique_email():
    for _ in range(50):
        e = fake.unique.email()
        if e not in existing_emails:
            existing_emails.add(e); return e
    e = f"user_{secrets.token_hex(6)}@gmail.com"
    existing_emails.add(e); return e

def unique_phone():
    for _ in range(50):
        p = fake.unique.phone_number()[:20]
        if p not in existing_phones:
            existing_phones.add(p); return p
    p = "07" + str(random.randint(100000000,999999999))
    existing_phones.add(p); return p

# ── Volume breakdown ──────────────────────────────────────────
# Total ~10,000 customers
#   30% Online    (email + phone + password)
#   20% Offline   (email only)
#   50% Anonymous (walk-in, no info kept — bulk of venue sales)

N_ONLINE    = 3000
N_OFFLINE   = 2000
N_ANONYMOUS = 5000   

print(f"Generating {N_ONLINE+N_OFFLINE+N_ANONYMOUS:,} customers...")

cursor.executemany(
    "INSERT INTO CUSTOMER (customer_name, email, phone_number, password_hash) VALUES (%s,%s,%s,%s)",
    [(fake.name(), unique_email(), unique_phone(), fake_bcrypt(fake.name()))
     for _ in range(N_ONLINE)]
)
connection.commit()
print(f"  ✓ {N_ONLINE:,} online customers")

cursor.executemany(
    "INSERT INTO CUSTOMER (customer_name, email) VALUES (%s,%s)",
    [(fake.name(), unique_email()) for _ in range(N_OFFLINE)]
)
connection.commit()
print(f"  ✓ {N_OFFLINE:,} offline (email kept) customers")

cursor.executemany(
    "INSERT INTO CUSTOMER (customer_name) VALUES (%s)",
    [('Walk-in Customer',)] * N_ANONYMOUS
)
connection.commit()
print(f"  ✓ {N_ANONYMOUS:,} anonymous customers")

cursor.execute("""
    SELECT
        CASE WHEN password_hash IS NOT NULL THEN 'Online'
             WHEN email IS NOT NULL         THEN 'Offline (email)'
             ELSE                                'Anonymous'
        END AS type, COUNT(*) AS total
    FROM CUSTOMER GROUP BY type ORDER BY total DESC
""")
print(f"\n{'Type':<22} {'Count':>7}")
print("─"*31)
for t, c in cursor.fetchall():
    print(f"  {t:<20} {c:>7,}")
cursor.execute("SELECT COUNT(*) FROM CUSTOMER")
print(f"  {'Total':<20} {cursor.fetchone()[0]:>7,}")
cursor.close()

Customer Statistics:
  Offline    6 customers
  Online     10 customers
  Password Hash Integrity: ✓ OK


True

In [ ]:
# ============================================================
# CELL — INSERT TICKET (multi-seat cart simulation)
# ============================================================
# Python groups seats into cart sessions (1-4 seats per cart).
# All seats in a cart share the same customer + paid_at,
# representing a single checkout. SQL receives individual rows.
# ============================================================

import random, uuid
from datetime import datetime, timedelta
from collections import defaultdict

cursor = connection.cursor(dictionary=True)

# Load all non-Draft events
cursor.execute("""
    SELECT e.event_id, e.status, e.event_date, e.sale_open_at, e.stadium_id
    FROM EVENT e WHERE e.status != 'Draft'
""")
events = cursor.fetchall()

# Load all available seats grouped by event
cursor.execute("""
    SELECT es.event_seat_id, es.event_id, es.zone_id, s.seat_type
    FROM EVENT_SEAT es
    JOIN SEAT s ON s.seat_id = es.seat_id
    WHERE es.status = 'Available'
    ORDER BY es.event_id, RAND()
""")
all_seats = cursor.fetchall()
seats_by_event = defaultdict(list)
for seat in all_seats:
    seats_by_event[seat['event_id']].append(seat)

# Load base prices per zone
cursor.execute("SELECT zone_id, base_price FROM EVENT_ZONE")
zone_prices = {r['zone_id']: float(r['base_price']) for r in cursor.fetchall()}

# Segment customers by account type
cursor.execute("SELECT customer_id, email, password_hash FROM CUSTOMER")
all_customers = cursor.fetchall()
online_ids    = [c['customer_id'] for c in all_customers if c['password_hash']]
offline_ids   = [c['customer_id'] for c in all_customers if c['email'] and not c['password_hash']]
anonymous_ids = [c['customer_id'] for c in all_customers if not c['email']]

# Segment box offices by channel type
cursor.execute("SELECT box_office_id, office_type FROM BOX_OFFICE")
bos      = cursor.fetchall()
online_bo = [b['box_office_id'] for b in bos if b['office_type'] == 'Online']
venue_bo  = [b['box_office_id'] for b in bos if b['office_type'] == 'Venue']
agent_bo  = [b['box_office_id'] for b in bos if b['office_type'] == 'Agent']
cursor.close()


# =============================================================
# HELPER FUNCTIONS
# =============================================================

def pick_box_office():
    """
    Randomly selects a box office channel.
    Distribution: 40% Online, 35% Agent, 25% Venue.
    """
    r = random.random()
    if r < 0.40: return random.choice(online_bo)
    if r < 0.75: return random.choice(agent_bo)
    return random.choice(venue_bo)


def pick_customer(bo_id):
    """
    Selects a customer based on the sales channel:
      Online  -> registered or email-only customers
      Venue / Agent -> 50% anonymous walk-in, 50% registered/email
    """
    if bo_id in online_bo:
        pool = online_ids + offline_ids
        return random.choice(pool) if pool else 1
    if random.random() < 0.50:
        return random.choice(anonymous_ids) if anonymous_ids else 1
    pool = offline_ids + online_ids
    return random.choice(pool) if pool else 1


def cart_size(bo_id):
    """
    Returns number of seats in one checkout session.
    Online channel: larger carts (2-4 seats, planned purchases).
    Venue counter:  smaller carts (1-2 seats, walk-in).
    Agent:          medium carts (1-3 seats).
    """
    if bo_id in online_bo:
        return random.choices([1, 2, 3, 4], weights=[20, 40, 25, 15])[0]
    if bo_id in venue_bo:
        return random.choices([1, 2, 3],    weights=[50, 35, 15])[0]
    return random.choices([1, 2, 3],        weights=[40, 40, 20])[0]


def calc_price(base, event_date, sale_open_at, purchase_time, sold_pct):
    """
    Applies dynamic pricing rules in order of priority:
      1. Early Bird  : purchased within 5 days of sale opening -> -10%
      2. Last Minute : purchased within 3 days of event        -> +15%
      3. High Demand : sold_pct >= 85%                         -> +20%
      4. Base Price  : no rule triggered
    Mirrors the pricing_config JSON logic stored in EVENT.
    """
    days_since_open = (purchase_time - sale_open_at).days if sale_open_at else 999
    days_to_event   = (event_date - purchase_time).days   if event_date   else 0

    if days_since_open <= 5:
        return round(base * 0.90, 2), "Early Bird: -10%"
    if days_to_event <= 3:
        return round(base * 1.15, 2), "Last Minute: +15%"
    if sold_pct >= 85:
        return round(base * 1.20, 2), f"High Demand: +20% ({sold_pct:.0f}% sold)"
    return round(base, 2), "Base Price"


def purchase_time(event_date, sale_open_at, bucket):
    """
    Generates a realistic purchase timestamp within the sale window.
    Bucket 'early'  : first 5 days after sale opens
    Bucket 'late'   : last 3 days before event
    Bucket 'middle' : anywhere in between
    """
    if not sale_open_at:
        sale_open_at = event_date - timedelta(days=60)
    window = max((event_date - sale_open_at).days, 1)
    if bucket == 'early':
        offset = random.randint(0, min(4, window - 1))
    elif bucket == 'late':
        offset = max(0, window - random.randint(1, 3))
    else:
        offset = random.randint(5, max(5, window - 4))
    base = sale_open_at + timedelta(days=offset)
    return base + timedelta(hours=random.randint(8, 22), minutes=random.randint(0, 59))


def get_fill_rate(status, event_date):
    """
    Returns the target seat fill rate for an event.
    SoldOut / Finished events have high fill rates.
    Future OnSale events vary by how soon they occur.
    """
    if status == 'SoldOut':  return 1.00
    if status == 'Finished': return random.uniform(0.82, 0.95)
    days_left = (event_date - datetime.now()).days
    if days_left > 60: return random.uniform(0.10, 0.25)
    if days_left > 14: return random.uniform(0.25, 0.50)
    return random.uniform(0.50, 0.70)


def bucket_weights(status, event_date):
    """
    Returns sampling weights for purchase timing buckets
    [early, middle, late]. Finished/SoldOut events show
    a more even distribution across the sale window.
    """
    if status in ('Finished', 'SoldOut'): return [0.35, 0.35, 0.30]
    days_left = (event_date - datetime.now()).days
    if days_left > 30: return [0.60, 0.40, 0.00]
    return [0.40, 0.30, 0.30]


# =============================================================
# MAIN LOOP — build ticket + seat update batches
# =============================================================

ticket_rows        = []   # rows to insert into TICKET
event_seat_updates = []   # (new_status, event_seat_id) pairs for EVENT_SEAT

for event in events:
    eid     = event['event_id']
    estatus = event['status']
    edate   = event['event_date']
    sopen   = event.get('sale_open_at')
    seats   = seats_by_event[eid]
    if not seats:
        continue

    n_fill    = int(len(seats) * get_fill_rate(estatus, edate))
    pool      = seats[:n_fill]
    total_cap = len(seats)
    bweights  = bucket_weights(estatus, edate)
    buckets   = ['early', 'middle', 'late']

    i = 0   # tracks position in pool for sold_pct calculation
    while i < len(pool):

        # ── Build one cart session ────────────────────────────
        bo_id     = pick_box_office()
        cust_id   = pick_customer(bo_id)
        n_seats   = min(cart_size(bo_id), len(pool) - i)
        bucket    = random.choices(buckets, weights=bweights)[0]
        paid_time = purchase_time(edate, sopen, bucket)

        # Determine ticket status for all seats in this cart
        if estatus in ('Finished', 'SoldOut'):
            t_status = 'Paid'
        else:
            r = random.random()
            t_status = 'Paid' if r < 0.85 else ('Cancelled' if r < 0.95 else 'Pending')

        # ── Add each seat in the cart as one TICKET row ───────
        for j in range(n_seats):
            seat     = pool[i]
            esid     = seat['event_seat_id']
            base     = zone_prices.get(seat['zone_id'], 25.00)
            sold_pct = (i / total_cap) * 100

            price, reason = calc_price(base, edate, sopen, paid_time, sold_pct)

            expires_at = datetime.now() + timedelta(minutes=10) if t_status == 'Pending' else None
            paid_at    = paid_time if t_status == 'Paid' else None
            qr_code    = str(uuid.uuid4()) if t_status == 'Paid' else None
            # 80% of seats at Finished events have been scanned at the gate
            is_scanned = 1 if (t_status == 'Paid' and estatus == 'Finished' and random.random() < 0.80) else 0

            ticket_rows.append((
                esid, cust_id, bo_id, t_status,
                price, reason, expires_at, paid_at, qr_code, is_scanned
            ))

            # Track which EVENT_SEAT rows need a status update
            if   t_status == 'Paid':    event_seat_updates.append(('Booked', esid))
            elif t_status == 'Pending': event_seat_updates.append(('Locked', esid))
            # Cancelled tickets leave the seat as Available — no update needed

            i += 1


# =============================================================
# STEP 1 — Insert TICKET rows
# =============================================================
# Triggers (trg_ticket_paid, trg_ticket_released) are bypassed
# here because executemany fires them 400k+ times, making the
# batch extremely slow. Seat and event statuses are updated
# manually in Steps 2 and 3 below.

cursor = connection.cursor()
cursor.executemany("""
    INSERT INTO TICKET
        (event_seat_id, customer_id, box_office_id, status, price_paid,
         price_reason, expires_at, paid_at, qr_code, is_scanned)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", ticket_rows)
connection.commit()
print(f"✓ Inserted {cursor.rowcount:,} tickets")


# =============================================================
# STEP 2 — Update EVENT_SEAT status
# =============================================================
# Bypasses trg_check_soldout (would fire 400k+ times in a loop).
# EVENT status is corrected in Step 3 instead.

cursor.executemany(
    "UPDATE EVENT_SEAT SET status = %s WHERE event_seat_id = %s",
    event_seat_updates
)
connection.commit()
print(f"✓ Updated {len(event_seat_updates):,} seat statuses")


# =============================================================
# STEP 3 — Sync EVENT status from seat inventory
# =============================================================
# Replaces what trg_check_soldout would have done during seeding.
# Marks any OnSale event as SoldOut if no Available seats remain.
# Finished / Draft statuses were set at insert time and are not changed.

cursor.execute("""
    UPDATE EVENT e
    SET    e.status = 'SoldOut'
    WHERE  e.status = 'OnSale'
      AND  NOT EXISTS (
               SELECT 1 FROM EVENT_SEAT es
               WHERE  es.event_id = e.event_id
                 AND  es.status   = 'Available'
           )
""")
connection.commit()
print(f"✓ Marked {cursor.rowcount} event(s) as SoldOut")


# =============================================================
# VERIFY — per-event summary
# =============================================================

cursor = connection.cursor(dictionary=True)
cursor.execute("""
    SELECT
        e.event_name,
        e.status,
        SUM(t.status = 'Paid')                                       AS paid,
        SUM(t.status = 'Cancelled')                                  AS cancelled,
        SUM(t.status = 'Pending')                                    AS pending,
        ROUND(SUM(t.status = 'Paid') / COUNT(es.event_seat_id) * 100, 1) AS sold_pct
    FROM EVENT e
    JOIN EVENT_SEAT es ON es.event_id      = e.event_id
    LEFT JOIN TICKET t ON t.event_seat_id  = es.event_seat_id
    WHERE e.status != 'Draft'
    GROUP BY e.event_id
    ORDER BY e.event_id
""")
print(f"\n{'Event':<42} {'Status':<10} {'Paid':>7} {'Cancel':>7} {'Pending':>8} {'Sold%':>6}")
print("─" * 87)
for r in cursor.fetchall():
    print(f"  {r['event_name']:<40} {r['status']:<10} "
          f"{r['paid']:>7,} {r['cancelled']:>7,} {r['pending']:>8,} {r['sold_pct']:>5}%")

# Multi-seat cart sanity check
cursor.execute("""
    SELECT
        COUNT(*)   AS carts,
        AVG(cnt)   AS avg_size,
        MAX(cnt)   AS max_size
    FROM (
        SELECT customer_id, paid_at, COUNT(*) AS cnt
        FROM   TICKET
        WHERE  status = 'Paid' AND paid_at IS NOT NULL
        GROUP  BY customer_id, paid_at
        HAVING cnt > 1
    ) multi
""")
r = cursor.fetchone()
print(f"\nMulti-seat carts: {r['carts']:,}  "
      f"avg size: {float(r['avg_size']):.1f}  "
      f"max size: {r['max_size']}")
cursor.close()
